# Back to the feature: an end-to-end paper recommender

_Feature Engineering_

**Compose the book's transforms into one pipeline — one-hot the categories, scale the numbers, normalize, and retrieve nearest neighbors.**

A content recommender is just "turn each item into a vector, then find the nearest vectors". All the work is in turning the item into a good vector — that is feature engineering — and in making the geometry of "nearest" meaningful — that is scaling and normalization.

       For an academic paper, the vector is built by concatenating the outputs of simple transforms: a one-hot / bag-of-words block for the fields of study, plus a handful of scaled numeric columns (year, citation count). Stack these side by side and you get one feature row per paper, i.e. a feature matrix $X$ with one row per item.

---

This notebook builds the pipeline up one step at a time. Run each cell, read the note above it, watch one over-scaled feature wreck the recommendations and scaling fix them, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Load the item catalog and a precision helper

We treat `load_wine` as a stand-in catalog: 178 "items" (papers), 13 numeric features, and a `topic` label (the wine class) that a good recommender should match. The `topic` is never fed into the features — it is only the answer key. The `precision_at_k` helper asks each item for its `k` nearest cosine neighbors (excluding itself) and measures the fraction that share its topic; higher means better recommendations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors

# Load real data. X = item feature matrix (178 items x 13 features).
# topic = the class label per item (0/1/2) = the "topic" a good recommender should
# match. We NEVER feed 'topic' into the features; it is only the answer key.
data = load_wine()
X, topic = data.data, data.target           # X: 178 x 13 ; topic: 178 labels
n_items = X.shape[0]
K = 5                                        # recommend 5 neighbors per item

def precision_at_k(features, k):
    # cosine distance; ask k+1 because the closest match is the item itself.
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(features)
    neigh = nn.kneighbors(features, return_distance=False)[:, 1:]  # drop self
    same_topic = (topic[neigh] == topic[:, None])                 # hit / miss grid
    return same_topic.mean()                  # mean precision@k over all items

### Step 2 — Visualize the raw feature scales

Before measuring any distances, look at how wide each of the 13 columns spreads. Plotted on a log axis (otherwise the small ones vanish), one feature — `proline` — towers about 1000x over the rest. That single over-scaled column is the problem the rest of the notebook fixes.

In [ ]:
# Show that the 13 feature columns live on totally different scales.
# We plot each column's spread (min..max range). 'proline' towers over all others.
col_ranges = X.max(axis=0) - X.min(axis=0)   # how wide each feature spreads
order = np.argsort(col_ranges)               # smallest -> largest spread

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(np.array(data.feature_names)[order], col_ranges[order], color="indianred")
ax.set_xscale("log")                         # log axis or you can't see the small ones
ax.set_title("STEP 2: raw feature scales differ by ~1000x (note the log x-axis)")
ax.set_xlabel("range (max - min) of each raw feature, log scale")
plt.tight_layout()
plt.show()

### Step 3 — Reproduce the problem on raw features

Run the recommender on the raw, unscaled matrix. Because `proline` is ~1000x bigger than everything else, cosine similarity is effectively "how close is proline?" and ignores the other 12 features. Neighbors land on the wrong topic, so precision@5 is low — that printed number is the baseline we want to beat.

In [ ]:
# Run the recommender on RAW features. Because 'proline' dominates, neighbors are
# mostly the WRONG topic -> low precision@5.
raw_p5 = precision_at_k(X, K)
print(f"PROBLEM (raw, unscaled): precision@5 = {raw_p5:.3f}")

### Step 4 — Scale and normalize, then re-inspect the scales

Two transforms fix the geometry. **StandardScaler** subtracts each column's mean and divides by its std, so every feature now spans a comparable range and none can dominate. **L2-normalize** then rescales each row to unit length, so cosine compares pure direction — the *mix* of features — which is what we actually want. The before/after bars confirm the scales are now even.

In [ ]:
# (a) StandardScale: subtract mean, divide by std -> every feature now has the
#     SAME spread, so none can dominate.
# (b) L2-normalize each row -> each item vector has length 1, so cosine compares
#     pure DIRECTION (the mix of features).
X_scaled = StandardScaler().fit_transform(X)     # (a) equalize column scales
X_eng = normalize(X_scaled, norm="l2", axis=1)   # (b) unit-length rows

eng_ranges = X_scaled.max(axis=0) - X_scaled.min(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].barh(np.array(data.feature_names)[order], col_ranges[order], color="indianred")
ax[0].set_xscale("log")
ax[0].set_title("STEP 4a: RAW scales (one feature dominates)")
ax[0].set_xlabel("range, log scale")
ax[1].barh(np.array(data.feature_names)[order], eng_ranges[order], color="seagreen")
ax[1].set_title("STEP 4b: SCALED — every feature now has a comparable range")
ax[1].set_xlabel("range after StandardScaler")
plt.tight_layout()
plt.show()

### Step 5 — Show the fix and sweep every cutoff

Run the same recommender on the scaled + normalized features: now every feature gets a vote, neighbors share the item's topic, and precision@5 jumps. To show the win is not an artifact of `k=5`, we sweep `k=1..10` and plot both curves plus a side-by-side precision@5 bar. This is the iterative "add a feature / scale it / re-measure" loop a recommender engineer runs over and over.

In [ ]:
# SAME recommender, now on scaled + L2-normalized features.
eng_p5 = precision_at_k(X_eng, K)
print(f"FIX (scaled + L2-normalized): precision@5 = {eng_p5:.3f}")

# Sweep k=1..10 to show the win holds at every cutoff.
ks = range(1, 11)
raw_curve = [precision_at_k(X, k) for k in ks]
eng_curve = [precision_at_k(X_eng, k) for k in ks]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), raw_curve, "o-", color="indianred", label="raw (unscaled)")
ax[0].plot(list(ks), eng_curve, "o-", color="seagreen", label="scaled + normalized")
ax[0].set_title("STEP 5a: precision@k — raw vs engineered (higher is better)")
ax[0].set_xlabel("k (number of recommendations)")
ax[0].set_ylabel("precision@k")
ax[0].set_ylim(0, 1.02)
ax[0].legend()
ax[1].bar(["raw", "engineered"], [raw_p5, eng_p5], color=["indianred", "seagreen"])
ax[1].set_title("STEP 5b: mean precision@5 — raw vs engineered")
ax[1].set_ylabel("precision@5")
ax[1].set_ylim(0, 1.02)
for i, v in enumerate([raw_p5, eng_p5]):
    ax[1].text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

print(f"PROBLEM (raw): {raw_p5:.3f}   →   FIX (engineered): {eng_p5:.3f}")

## Reference implementation — scikit-learn

### Step 1 — Round 1: one-hot the fields of study

We load the Microsoft Academic Graph papers, each with its space-joined fields of study and a few numeric columns. For the first recommender we use only the categorical fields, one-hot encoded into a sparse 0/1 matrix with `CountVectorizer(binary=True)` — a bag of category tokens per paper.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix

# Microsoft Academic Graph papers, prepared as in Chapter 9.
# Data: github.com/alicezheng/feature-engineering-book (Chapter 9).
# Each row is a paper with its fields of study and a few numeric features.
papers = pd.read_csv('mag_papers.csv')
# columns: 'paper_id', 'title', 'fields_of_study' (space-joined string),
#          'year', 'citation_count'

# ROUND 1: fields of study only (one-hot / bag-of-words)
# Each paper's fields are a bag of category tokens; one-hot encode them.
fields_vec = CountVectorizer(binary=True, token_pattern=r'[^ ]+')
X_fields = fields_vec.fit_transform(papers['fields_of_study'])   # sparse 0/1

### Step 2 — Build the cosine nearest-neighbor recommender

Two small helpers wire up the retrieval. `build_recommender` L2-normalizes the rows and fits a cosine nearest-neighbor index; `recommend` queries it, drops the query itself (always its own closest match), and prints the rest. Running it on the fields-only matrix gives our first set of recommendations.

In [ ]:
def build_recommender(X, k=5):
    """L2-normalize rows, then fit a cosine nearest-neighbor index."""
    Xn = normalize(X, norm='l2', axis=1)          # unit-length rows
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine')
    nn.fit(Xn)
    return nn, Xn

def recommend(nn, Xn, query_idx, titles, k=5):
    dist, idx = nn.kneighbors(Xn[query_idx], n_neighbors=k + 1)
    # neighbor 0 is the query itself; drop it
    for d, j in zip(dist[0][1:], idx[0][1:]):
        print(f"  cos={1 - d:.3f}  {titles.iloc[j]}")

nn1, Xn1 = build_recommender(X_fields)
print("ROUND 1 (fields only):")
recommend(nn1, Xn1, query_idx=0, titles=papers['title'])

### Step 3 — Round 2: add scaled numeric features and re-measure

Now the iterative step. Raw `year` and `citation_count` live on a completely different scale than the 0/1 field flags, so we standardize them before appending — otherwise they would dominate the distance. We stack the scaled numerics onto the one-hot block, rebuild the recommender, and compare the two rounds' neighbors by hand. Feature engineering is this loop: add a feature, rescale, re-measure, keep what improves the recommendations.

In [ ]:
# ROUND 2: add SCALED numeric features, then re-measure.
# Raw year/citations live on a totally different scale than the 0/1 flags,
# so we standardize them before appending. This is the key iterative step.
num = papers[['year', 'citation_count']].to_numpy(dtype=float)
num_scaled = StandardScaler().fit_transform(num)          # mean 0, var 1
X_full = hstack([X_fields, csr_matrix(num_scaled)]).tocsr()

nn2, Xn2 = build_recommender(X_full)
print("ROUND 2 (fields + SCALED year & citations):")
recommend(nn2, Xn2, query_idx=0, titles=papers['title'])

# Feature engineering is ITERATIVE: add a feature, rescale, re-measure.
# Compare the two rounds' neighbors by hand and keep what improves them.
# (Optional next rounds: TfidfTransformer() to weight rare fields,
#  or TruncatedSVD/PCA to compress the wide sparse one-hot matrix.)

## Visualize the data & results

_On a bundled item catalog, does scaling + L2-normalizing the features before cosine nearest-neighbor retrieval return more same-class items (higher precision@k) than using RAW features?_

### Step 1 — Set up the catalog and a precision@k helper

We use the wine dataset again as an item catalog and define `precision_at_k`: it L2-normalizes the rows, retrieves each item's `k` nearest cosine neighbors, drops the item itself, and returns the average fraction of neighbors sharing the item's class. This is our single quality number for same-class retrieval.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors

# Bundled dataset as a stand-in ITEM CATALOG: 178 wines, 13 features, 3 classes.
X, y = load_wine(return_X_y=True)
k = 5

def precision_at_k(features, labels, k):
    """For each item, retrieve k nearest neighbors by cosine similarity;
    measure the fraction that share the item's class (same-class retrieval)."""
    Xn = normalize(features, norm='l2', axis=1)        # unit-length rows
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(Xn)
    _, idx = nn.kneighbors(Xn)
    hits = []
    for i in range(len(labels)):
        neigh = idx[i][1:]                              # drop the item itself
        hits.append(np.mean(labels[neigh] == labels[i]))
    return float(np.mean(hits))

### Step 2 — Compare raw vs scaled retrieval

We score precision@5 twice: once on the raw features (where `proline` dominates the distance) and once after standardizing every column then L2-normalizing the rows. The printed pair shows scaling before a distance-based retrieval lifts same-class precision substantially.

In [ ]:
# RAW: features on wildly different scales (e.g. 'proline' is in the hundreds).
raw_p = precision_at_k(X, y, k)

# SCALED + normalized: standardize every column first, then L2-normalize rows.
X_scaled = StandardScaler().fit_transform(X)
scaled_p = precision_at_k(X_scaled, y, k)

print([round(raw_p, 2), round(scaled_p, 2)])
# -> [0.76, 0.93]   scaling before a distance-based retrieval helps a lot

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You one-hot encode 5000 fields of study and append one raw "citation count" column (values up to ~50000), then retrieve nearest neighbors by cosine similarity. Every recommendation is a hugely-cited paper on an unrelated topic. What went wrong and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the magnitude of the citation column to the 0/1 field columns. — _The field flags are 0 or 1; the citation count is in the tens of thousands, so its square swamps the sum of squares in the vector length._
- Recognize the distance is now dominated by one feature. — _Cosine similarity is driven almost entirely by the citation column, so papers look similar when their citation counts are similar, regardless of topic._
- Scale every column before measuring distance. — _Standardizing (or min-max scaling) the citation column to the same order of magnitude as the 0/1 flags lets the fields of study contribute to the similarity again._

**Answer:** The unscaled citation column dominates the distance: with values up to 50000 its square dwarfs the 0/1 field indicators, so cosine similarity ranks papers by citation magnitude rather than topic. Fix: scale/normalize every feature first — standardize the citation column to mean 0, variance 1 (or min-max to $[0,1]$) so it sits on the same scale as the field flags — then L2-normalize each row and re-run nearest neighbors. Now the shared fields of study drive the recommendations.

</details>

**Problem 2.** Your first recommender uses only fields of study and gives mediocre results. A teammate says "just switch from cosine to Euclidean distance and it'll be fixed." Per the book's Chapter 9 philosophy, is that the right move?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what actually drives a content recommender's quality. — _The book stresses that the features and their scaling matter far more than the choice of similarity metric._
- Think about the iterative feature-engineering loop. — _Chapter 9 improves the recommender by adding raw features (year, citations) and re-scaling them, then re-inspecting neighbors — not by swapping metrics._
- Decide the next action. — _Adding and scaling informative features addresses the root cause; changing the metric on the same weak feature set rarely helps much._

**Answer:** No — swapping the metric is the wrong lever. The book's lesson is that feature engineering is iterative and that features and their scaling drive quality far more than the distance metric. The right move is to add more raw features (publication year, citation/reference counts), scale and normalize them so none dominates, re-run nearest neighbors, and inspect the recommendations. If that still under-performs, consider tf-idf weighting of the fields or PCA to compress the wide one-hot matrix. Iterate: add a feature, rescale, re-measure.

</details>